# HireSignal — QLoRA fine-tuning of Qwen3-4B-Instruct-2507 (Kaggle)

Settings: **Accelerator GPU T4 x2 (or P100)**, **Internet ON** (downloads the base model from Hugging Face).
Input: the PRIVATE Kaggle dataset `hiresignal-finetune` made from `hiresignal_finetune_bundle.zip`.

Trains ONLY on `finetuning/data/sft_train.jsonl` (synthetic train split). Validation prompts are used only for the final comparison. The sealed test split is not in the bundle.

In [ ]:
!pip install -q -U "transformers>=4.51,<6" "peft>=0.13" "bitsandbytes>=0.43" accelerate pydantic
import shutil, os
src = '/kaggle/input/hiresignal-finetune'
# the dataset may be the zip itself or its extracted content
zips = [os.path.join(r, f) for r, _, fs in os.walk(src) for f in fs if f.endswith('.zip')]
if zips:
    shutil.unpack_archive(zips[0], '/kaggle/working/hs')
else:
    shutil.copytree(src, '/kaggle/working/hs', dirs_exist_ok=True)
os.chdir('/kaggle/working/hs'); print(sorted(os.listdir('.')))

In [ ]:
import json, hashlib
m = json.load(open('finetuning/data/manifest.json'))
for f, info in m['files'].items():
    h = hashlib.sha256(open('finetuning/data/' + f, 'rb').read()).hexdigest()
    print(f, 'OK' if h == info['sha256'] else 'SHA MISMATCH', {k: v for k, v in info.items() if k != 'sha256'})
print(json.dumps(m['leakage_audit'], indent=1))

## 1. Baseline: untuned Qwen3-4B on the 202 val requirements (same 4-bit setup)
Run this BEFORE training so the comparison uses identical conditions. ~15-25 min on T4.

In [ ]:
!python -m finetuning.eval_val --run-id base --out /kaggle/working/eval

## 2. QLoRA training (train split only)
Resumable: re-running the cell continues from the last checkpoint. ~30-60 min on T4.

In [ ]:
!python -m finetuning.train_qlora --out /kaggle/working/qwen3-4b-hs-lora --epochs 2

## 3. Fine-tuned adapter on the same 202 val requirements

In [ ]:
!python -m finetuning.eval_val --run-id lora --adapter /kaggle/working/qwen3-4b-hs-lora/adapter --out /kaggle/working/eval

In [ ]:
import json
for r in ('base', 'lora'):
    x = json.load(open(f'/kaggle/working/eval/{r}/metrics.json'))
    f = x['final']
    print(r, 'macro-F1 %.3f' % f['macro_f1'], 'false-MET %.3f' % f['false_met_rate'], 'JSON %.3f' % x['json_schema_validity'], 'quote-valid', x['evidence']['raw_quote_validity'], 'UNCERTAIN recall %.3f' % f['per_class']['UNCERTAIN']['recall'])

## 4. Save outputs
Download `/kaggle/working/results.zip` (adapter + train_meta.json + eval metrics/predictions). Do NOT publish the adapter or data publicly without the team's agreement. Do not run on the sealed test split; the test run is done once by the team after selection.

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/results', 'zip', '/kaggle/working', 'qwen3-4b-hs-lora/adapter')
shutil.make_archive('/kaggle/working/eval_results', 'zip', '/kaggle/working', 'eval')
shutil.copy('/kaggle/working/qwen3-4b-hs-lora/train_meta.json', '/kaggle/working/train_meta.json')
print('done')